<a href="https://colab.research.google.com/github/analystnilava/codtech_task_repo/blob/codtech_task/TASK2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get update -qq
!apt-get install -y openjdk-11-jdk-headless

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  openjdk-11-jre-headless
Suggested packages:
  openjdk-11-demo openjdk-11-source libnss-mdns fonts-dejavu-extra
  fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  | fonts-wqy-zenhei fonts-indic
The following NEW packages will be installed:
  openjdk-11-jdk-headless openjdk-11-jre-headless
0 upgraded, 2 newly installed, 0 to remove and 141 not upgraded.
Need to get 116 MB of archives.
After this operation, 258 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 openjdk-11-jre-headless amd64 11.0.30+7-1ubuntu1~22.04 [42.6 MB]
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates/main amd6

In [ ]:
!wget -q https://archive.apache.org/dist/spark/spark-3.4.4/spark-3.4.4-bin-hadoop3.tgz
!tar xf spark-3.4.4-bin-hadoop3.tgz

In [ ]:
!pip install -q findspark

In [ ]:
import os
import findspark

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.4.4-bin-hadoop3"

findspark.init(os.environ["SPARK_HOME"])

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("colab_spark") \
    .getOrCreate()

spark

### upload data


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving BigMart Sales.csv to BigMart Sales.csv


### load data

In [ ]:
df1 = spark.read.format('csv').option('inferSchema', True).option('header', True).load('BigMart Sales.csv')


In [ ]:
df1.show()

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|          FDA15|        9.3|         Low Fat|    0.016047301|               Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|Supermarket Type1|         3735.138|
|          DRC01|       5.92|         Regular|    0.019278216|         Soft Drinks| 48.2692|           OUT018|                     2009|     Medium|              Tier 3|Superma

In [ ]:
df1.printSchema()

root
 |-- Item_Identifier: string (nullable = true)
 |-- Item_Weight: double (nullable = true)
 |-- Item_Fat_Content: string (nullable = true)
 |-- Item_Visibility: double (nullable = true)
 |-- Item_Type: string (nullable = true)
 |-- Item_MRP: double (nullable = true)
 |-- Outlet_Identifier: string (nullable = true)
 |-- Outlet_Establishment_Year: integer (nullable = true)
 |-- Outlet_Size: string (nullable = true)
 |-- Outlet_Location_Type: string (nullable = true)
 |-- Outlet_Type: string (nullable = true)
 |-- Item_Outlet_Sales: double (nullable = true)



In [ ]:
# schema ddl
ddl_schema = """
Item_Identifier STRING,
Item_Weight string,
Item_Fat_Content STRING,
Item_Visibility DOUBLE,
Item_Type STRING,
Item_MRP DOUBLE,
Outlet_Identifier STRING,
Outlet_Establishment_Year INT,
Outlet_Size STRING,
Outlet_Location_Type STRING,
Outlet_Type STRING,
Item_Outlet_Sales DOUBLE
"""


### Explore Dataset

In [ ]:
df1.printSchema()

root
 |-- Item_Identifier: string (nullable = true)
 |-- Item_Weight: double (nullable = true)
 |-- Item_Fat_Content: string (nullable = true)
 |-- Item_Visibility: double (nullable = true)
 |-- Item_Type: string (nullable = true)
 |-- Item_MRP: double (nullable = true)
 |-- Outlet_Identifier: string (nullable = true)
 |-- Outlet_Establishment_Year: integer (nullable = true)
 |-- Outlet_Size: string (nullable = true)
 |-- Outlet_Location_Type: string (nullable = true)
 |-- Outlet_Type: string (nullable = true)
 |-- Item_Outlet_Sales: double (nullable = true)



In [ ]:
df2 = spark.read.format('csv').schema(ddl_schema).option('header', True).load('BigMart Sales.csv')

df2.printSchema()

root
 |-- Item_Identifier: string (nullable = true)
 |-- Item_Weight: string (nullable = true)
 |-- Item_Fat_Content: string (nullable = true)
 |-- Item_Visibility: double (nullable = true)
 |-- Item_Type: string (nullable = true)
 |-- Item_MRP: double (nullable = true)
 |-- Outlet_Identifier: string (nullable = true)
 |-- Outlet_Establishment_Year: integer (nullable = true)
 |-- Outlet_Size: string (nullable = true)
 |-- Outlet_Location_Type: string (nullable = true)
 |-- Outlet_Type: string (nullable = true)
 |-- Item_Outlet_Sales: double (nullable = true)



### DATA CLANING

In [ ]:
df1 = df1.fillna({
    "Item_Fat_Content": "Unknown",
    "Item_Type": "Unknown",
    "Outlet_Size": "Medium",
    "Outlet_Location_Type": "Unknown",
    "Outlet_Type": "Unknown"
})

df1.show()

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|          FDA15|        9.3|         Low Fat|    0.016047301|               Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|Supermarket Type1|         3735.138|
|          DRC01|       5.92|         Regular|    0.019278216|         Soft Drinks| 48.2692|           OUT018|                     2009|     Medium|              Tier 3|Superma

### Convert Categorical → Numeric

In [ ]:
from pyspark.ml.feature import StringIndexer

from pyspark.ml.feature import StringIndexer

categorical_cols = [
    "Item_Fat_Content",
    "Item_Type",
    "Outlet_Size",
    "Outlet_Location_Type",
    "Outlet_Type"
]

for col_name in categorical_cols:
    index_col = col_name + "_index"

    # 🔥 FIX: Drop column if already exists
    if index_col in df1.columns:
        df1 = df1.drop(index_col)

    indexer = StringIndexer(
        inputCol=col_name,
        outputCol=index_col,
        handleInvalid="keep"
    )

    df1 = indexer.fit(df1).transform(df1)

df1.show(5)

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----------------------+---------------+-----------------+--------------------------+-----------------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|Item_Fat_Content_index|Item_Type_index|Outlet_Size_index|Outlet_Location_Type_index|Outlet_Type_index|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----------------------+---------------+-----------------+--------------------------+-----------------+
|          FDA15|        9.3|         Low Fat|    0.016047301|              

### Feature Engineering

In [ ]:
from pyspark.ml.feature import VectorAssembler

feature_cols = [
    "Item_Weight",
    "Item_Visibility",
    "Item_MRP",
    "Item_Fat_Content_index",
    "Item_Type_index",
    "Outlet_Size_index",
    "Outlet_Location_Type_index",
    "Outlet_Type_index"
]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

df_final = assembler.transform(df1)
df_final.show(5)

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----------------------+---------------+-----------------+--------------------------+-----------------+--------------------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|Item_Fat_Content_index|Item_Type_index|Outlet_Size_index|Outlet_Location_Type_index|Outlet_Type_index|            features|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----------------------+---------------+-----------------+--------------------------+-----------------+--------------------+
|          FD

### Train-Test Split

In [ ]:
train_data, test_data = df_final.randomSplit([0.8, 0.2], seed=42)

HANDEL NULL VALUE


In [ ]:
from pyspark.sql.functions import mean

# mean for numeric
mean_weight = df1.select(mean("Item_Weight")).collect()[0][0]

df1 = df1.fillna({
    "Item_Weight": mean_weight,
    "Item_Visibility": 0.0,
    "Item_MRP": 0.0,
    "Item_Fat_Content_index": 0,
    "Item_Type_index": 0,
    "Outlet_Size_index": 0,
    "Outlet_Location_Type_index": 0,
    "Outlet_Type_index": 0
})

## Train Model

In [ ]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=[
        "Item_Weight",
        "Item_Visibility",
        "Item_MRP",
        "Item_Fat_Content_index",
        "Item_Type_index",
        "Outlet_Size_index",
        "Outlet_Location_Type_index",
        "Outlet_Type_index"
    ],
    outputCol="features",
    handleInvalid="skip"   # 🔥 THIS LINE FIXES YOUR ERROR
)

df_final = assembler.transform(df1)

In [ ]:
df_final = df_final.dropna(subset=["features", "Item_Outlet_Sales"])

In [ ]:
from pyspark.ml.regression import LinearRegression

train_data, test_data = df_final.randomSplit([0.8, 0.2], seed=42)

lr = LinearRegression(
    featuresCol="features",
    labelCol="Item_Outlet_Sales"
)

model = lr.fit(train_data)

print("Model trained successfully ")

Model trained successfully 


In [ ]:
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="skip"
)

df_final = assembler.transform(df1)

### Prediction

In [ ]:
predictions = model.transform(test_data)

predictions.select(
    "Item_Outlet_Sales",
    "prediction"
).show(5)

+-----------------+------------------+
|Item_Outlet_Sales|        prediction|
+-----------------+------------------+
|        2552.6772|2300.3632203717443|
|         4913.604|2859.4801602937023|
|        4422.2436| 2617.888889199887|
|        2406.2012|  3278.56143155843|
|        7033.5112|2837.6500285199118|
+-----------------+------------------+
only showing top 5 rows



### Evaluation

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(
    labelCol="Item_Outlet_Sales",
    predictionCol="prediction",
    metricName="r2"
)

r2 = evaluator.evaluate(predictions)

print("R2 Score:", r2)

R2 Score: 0.3265238367692136
